# SAP Report Refresh

Filters exported BSNY and SANCAP SAP reports, identifies new entity-scoped Document IDs, runs duplicate and YTD controls, then appends validated rows to the SAP master through Excel COM.

In [17]:
from __future__ import annotations

import datetime as dt
import re
from collections import Counter
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any, Iterable

import pythoncom
import win32com.client as win32

# Input and output files for the monthly SAP refresh.
REPORTING_YEAR = dt.date.today().year
MASTER_FILE = Path(
    "samples/Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED - Try SAP.xlsx"
)
BSNY_SAP_FILE = Path("samples/BSNY - SAP & Concur Repoort May 2025 - June 2026.xlsx")
SANCAP_SAP_FILE = Path("samples/SanCap - VIM STATUS REPORT 2026-07-01 8.49 AM.xlsx")
OUTPUT_DIRECTORY = Path("outputs")
PUBLISHED_OUTPUT_FILE = OUTPUT_DIRECTORY / "SAP_Combined_Master_Refreshed.xlsx"

MASTER_SHEET_NAMES = ("SAP Report", "SAP Invoices Report")
BSNY_SHEET_NAME = "SAP"
ENTITY_HEADER = "Entity"
POSTING_DATE_HEADER = "Posting Date"
DOCUMENT_ID_HEADER = "Document ID"
GROSS_AMOUNT_HEADER = "Gross Amount in Local Currency"
COST_CENTER_HEADER = "Cost Center"

# Populate from the client's approved scope before running the refresh.
IN_SCOPE_COST_CENTERS: set[str] = set()

# Map master headers to their matching source headers when the wording differs.
# Entity is supplied by the source configuration and must be present in the master.
SOURCE_TO_MASTER_HEADERS: dict[str, str] = {
    "Posting Date": "Posting Date",
    "Document ID": "Document ID",
    "Gross Amount in Local Currency": "Gross Amount in Local Currency",
    "Cost Center": "Cost Center",
}

XL_UP = -4162
XL_CALCULATION_MANUAL = -4135
XL_CALCULATION_AUTOMATIC = -4105
XL_SHEET_VERY_HIDDEN = 2


@dataclass(frozen=True)
class SourceRow:
    entity: str
    document_id: str | None
    posting_date: dt.date | None
    values: dict[str, Any]
    source_name: str
    source_row_number: int

    @property
    def key(self) -> tuple[str, str] | None:
        if self.document_id is None:
            return None
        return (self.entity, self.document_id)


def normalize_text(value: Any) -> str:
    """Return an uppercase, whitespace-normalized value for matching."""
    return " ".join(str(value or "").strip().split()).upper()


def normalize_header(value: Any) -> str:
    return normalize_text(value)


def normalize_document_id(value: Any) -> str | None:
    """Normalize Excel IDs without accidentally retaining a numeric .0 suffix."""
    if value is None or isinstance(value, bool):
        return None

    text = str(value).strip()
    if not text:
        return None

    try:
        decimal_value = Decimal(text)
    except InvalidOperation:
        return text.upper()

    if decimal_value == decimal_value.to_integral_value():
        return str(decimal_value.quantize(Decimal("1")))
    return format(decimal_value.normalize(), "f")


def coerce_excel_date(value: Any) -> dt.date | None:
    """Support COM date values, Excel serial dates, and common report date text."""
    if value is None or value == "":
        return None
    if isinstance(value, dt.datetime):
        return value.date()
    if isinstance(value, dt.date):
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        try:
            return (dt.datetime(1899, 12, 30) + dt.timedelta(days=float(value))).date()
        except (OverflowError, ValueError):
            return None

    text = str(value).strip()
    for pattern in ("%m/%d/%Y", "%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y"):
        try:
            return dt.datetime.strptime(text, pattern).date()
        except ValueError:
            continue
    return None


def read_headers(worksheet: Any) -> dict[str, int]:
    last_column = worksheet.Cells(1, worksheet.Columns.Count).End(-4159).Column
    raw_headers = worksheet.Range(worksheet.Cells(1, 1), worksheet.Cells(1, last_column)).Value2
    values = raw_headers[0] if isinstance(raw_headers, tuple) else (raw_headers,)

    headers: dict[str, int] = {}
    duplicates: set[str] = set()
    for column_number, value in enumerate(values, start=1):
        header = normalize_header(value)
        if not header:
            continue
        if header in headers:
            duplicates.add(header)
        headers[header] = column_number

    if duplicates:
        raise ValueError(f"Duplicate headers in {worksheet.Name}: {sorted(duplicates)}")
    return headers


def required_column(headers: dict[str, int], header: str, worksheet_name: str) -> int:
    column = headers.get(normalize_header(header))
    if column is None:
        raise ValueError(f"Required header {header!r} was not found in worksheet {worksheet_name!r}.")
    return column


def last_used_row(worksheet: Any) -> int:
    return worksheet.Cells(worksheet.Rows.Count, 1).End(XL_UP).Row


def worksheet_rows(worksheet: Any, entity: str, source_name: str, reporting_year: int) -> list[SourceRow]:
    """Bulk-read one SAP worksheet and retain rows posted in the reporting year."""
    headers = read_headers(worksheet)
    posting_date_column = required_column(headers, POSTING_DATE_HEADER, worksheet.Name)
    document_id_column = required_column(headers, DOCUMENT_ID_HEADER, worksheet.Name)
    final_row = last_used_row(worksheet)
    final_column = max(headers.values())
    if final_row < 2:
        return []

    raw_rows = worksheet.Range(worksheet.Cells(2, 1), worksheet.Cells(final_row, final_column)).Value2
    if not isinstance(raw_rows, tuple):
        raw_rows = (raw_rows,)
    if raw_rows and not isinstance(raw_rows[0], tuple):
        raw_rows = (raw_rows,)

    retained: list[SourceRow] = []
    for row_number, raw_row in enumerate(raw_rows, start=2):
        posting_date = coerce_excel_date(raw_row[posting_date_column - 1])
        if posting_date is None or posting_date.year != reporting_year:
            continue

        values = {
            header: raw_row[column - 1]
            for header, column in headers.items()
        }
        retained.append(
            SourceRow(
                entity=normalize_text(entity),
                document_id=normalize_document_id(raw_row[document_id_column - 1]),
                posting_date=posting_date,
                values=values,
                source_name=source_name,
                source_row_number=row_number,
            )
        )
    return retained


def classify_source_rows(
    source_rows: Iterable[SourceRow], existing_keys: set[tuple[str, str]]
) -> dict[str, list[SourceRow]]:
    """Assign every in-year source row one mutually exclusive lookup status."""
    source_rows = list(source_rows)
    key_counts = Counter(row.key for row in source_rows if row.key is not None)
    result = {"new": [], "existing": [], "missing_id": [], "duplicate_in_source": []}

    for row in source_rows:
        if row.key is None:
            result["missing_id"].append(row)
        elif key_counts[row.key] > 1:
            result["duplicate_in_source"].append(row)
        elif row.key in existing_keys:
            result["existing"].append(row)
        else:
            result["new"].append(row)
    return result


print(f"SAP refresh configured for reporting year {REPORTING_YEAR}.")

SAP refresh configured for reporting year 2026.


In [18]:
import csv
import shutil
from collections import defaultdict


@dataclass(frozen=True)
class MasterRow:
    key: tuple[str, str] | None
    posting_date: dt.date | None
    cost_center: str | None
    gross_amount: Decimal
    row_number: int


def normalize_cost_center(value: Any) -> str | None:
    return normalize_document_id(value)


def decimal_amount(value: Any) -> Decimal:
    if value is None or value == "":
        return Decimal("0")
    try:
        return Decimal(str(value).replace(",", "").strip())
    except (InvalidOperation, ValueError):
        raise ValueError(f"Gross amount {value!r} is not numeric.") from None


def last_used_row_in_column(worksheet: Any, column: int) -> int:
    return worksheet.Cells(worksheet.Rows.Count, column).End(XL_UP).Row


def master_rows(worksheet: Any) -> tuple[dict[str, int], list[MasterRow]]:
    headers = read_headers(worksheet)
    entity_column = required_column(headers, ENTITY_HEADER, worksheet.Name)
    document_id_column = required_column(headers, DOCUMENT_ID_HEADER, worksheet.Name)
    posting_date_column = required_column(headers, POSTING_DATE_HEADER, worksheet.Name)
    cost_center_column = required_column(headers, COST_CENTER_HEADER, worksheet.Name)
    gross_amount_column = required_column(headers, GROSS_AMOUNT_HEADER, worksheet.Name)
    final_row = last_used_row_in_column(worksheet, document_id_column)
    final_column = max(headers.values())
    if final_row < 2:
        return headers, []

    raw_rows = worksheet.Range(worksheet.Cells(2, 1), worksheet.Cells(final_row, final_column)).Value2
    if not isinstance(raw_rows, tuple):
        raw_rows = (raw_rows,)
    if raw_rows and not isinstance(raw_rows[0], tuple):
        raw_rows = (raw_rows,)

    rows: list[MasterRow] = []
    for row_number, raw_row in enumerate(raw_rows, start=2):
        document_id = normalize_document_id(raw_row[document_id_column - 1])
        entity = normalize_text(raw_row[entity_column - 1])
        key = (entity, document_id) if entity and document_id else None
        rows.append(
            MasterRow(
                key=key,
                posting_date=coerce_excel_date(raw_row[posting_date_column - 1]),
                cost_center=normalize_cost_center(raw_row[cost_center_column - 1]),
                gross_amount=decimal_amount(raw_row[gross_amount_column - 1]),
                row_number=row_number,
            )
        )
    return headers, rows


def entity_counts_and_totals(
    rows: Iterable[SourceRow | MasterRow], reporting_year: int, source_rows: bool
) -> tuple[dict[str, int], dict[str, Decimal]]:
    counts: dict[str, int] = defaultdict(int)
    totals: dict[str, Decimal] = defaultdict(lambda: Decimal("0"))
    scoped_cost_centers = {normalize_cost_center(value) for value in IN_SCOPE_COST_CENTERS}

    for row in rows:
        if source_rows:
            assert isinstance(row, SourceRow)
            entity = row.entity
            posting_date = row.posting_date
            cost_center = normalize_cost_center(row.values.get(normalize_header(COST_CENTER_HEADER)))
            gross_amount = decimal_amount(row.values.get(normalize_header(GROSS_AMOUNT_HEADER)))
        else:
            assert isinstance(row, MasterRow)
            if row.key is None:
                continue
            entity = row.key[0]
            posting_date = row.posting_date
            cost_center = row.cost_center
            gross_amount = row.gross_amount

        if posting_date is None or posting_date.year != reporting_year:
            continue
        counts[entity] += 1
        if cost_center in scoped_cost_centers:
            totals[entity] += gross_amount
    return dict(counts), dict(totals)


def validate_structure(master_headers: dict[str, int], source_headers: dict[str, int]) -> None:
    required_column(master_headers, ENTITY_HEADER, MASTER_SHEET_NAME)
    for master_header, source_header in SOURCE_TO_MASTER_HEADERS.items():
        required_column(master_headers, master_header, MASTER_SHEET_NAME)
        required_column(source_headers, source_header, "source report")


def control_failures(
    source_rows: list[SourceRow],
    classification: dict[str, list[SourceRow]],
    existing_master_rows: list[MasterRow],
) -> list[str]:
    failures: list[str] = []
    duplicate_master_keys = [key for key, count in Counter(row.key for row in existing_master_rows if row.key).items() if count > 1]
    if duplicate_master_keys:
        failures.append(f"Master contains {len(duplicate_master_keys)} duplicate Entity + Document ID key(s).")
    if classification["missing_id"]:
        failures.append(f"Source contains {len(classification['missing_id'])} row(s) without a Document ID.")
    if classification["duplicate_in_source"]:
        failures.append(f"Source contains {len(classification['duplicate_in_source'])} duplicate Entity + Document ID row(s).")

    source_counts, source_totals = entity_counts_and_totals(source_rows, REPORTING_YEAR, source_rows=True)
    master_counts, master_totals = entity_counts_and_totals(existing_master_rows, REPORTING_YEAR, source_rows=False)
    entities = set(source_counts) | set(master_counts)
    for entity in sorted(entities):
        if source_counts.get(entity, 0) < master_counts.get(entity, 0):
            failures.append(
                f"{entity}: source YTD row count ({source_counts.get(entity, 0)}) is lower than master YTD row count ({master_counts.get(entity, 0)})."
            )
        if source_totals.get(entity, Decimal("0")) < master_totals.get(entity, Decimal("0")):
            failures.append(
                f"{entity}: source in-scope YTD gross total ({source_totals.get(entity, Decimal('0'))}) is lower than master ({master_totals.get(entity, Decimal('0'))})."
            )
    return failures


def next_available_output_path(path: Path) -> Path:
    if not path.exists():
        return path
    for suffix in range(1, 10_000):
        candidate = path.with_stem(f"{path.stem}_{suffix}")
        if not candidate.exists():
            return candidate
    raise RuntimeError("Could not allocate an unused output filename.")


def write_audit(
    classification: dict[str, list[SourceRow]],
    failures: list[str],
    source_rows: list[SourceRow],
    existing_master_rows: list[MasterRow],
) -> Path:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    audit_path = OUTPUT_DIRECTORY / f"sap_refresh_audit_{timestamp}.csv"
    source_counts, source_totals = entity_counts_and_totals(source_rows, REPORTING_YEAR, source_rows=True)
    master_counts, master_totals = entity_counts_and_totals(existing_master_rows, REPORTING_YEAR, source_rows=False)

    with audit_path.open("w", newline="", encoding="utf-8") as audit_file:
        writer = csv.writer(audit_file)
        writer.writerow(["section", "entity", "metric", "value"])
        writer.writerow(["run", "", "reporting_year", REPORTING_YEAR])
        writer.writerow(["run", "", "status", "FAILED" if failures else "PASSED"])
        for failure in failures:
            writer.writerow(["failure", "", "message", failure])
        for entity in sorted(set(source_counts) | set(master_counts)):
            writer.writerow(["source", entity, "ytd_row_count", source_counts.get(entity, 0)])
            writer.writerow(["master", entity, "ytd_row_count", master_counts.get(entity, 0)])
            writer.writerow(["source", entity, "in_scope_gross_total", source_totals.get(entity, Decimal("0"))])
            writer.writerow(["master", entity, "in_scope_gross_total", master_totals.get(entity, Decimal("0"))])
        for status, rows in classification.items():
            for row in rows:
                writer.writerow([status, row.entity, "source_row", f"{row.source_name}:{row.source_row_number}; Document ID={row.document_id or ''}"])
    return audit_path


def resize_containing_table(worksheet: Any, old_last_row: int, new_last_row: int) -> None:
    for index in range(1, worksheet.ListObjects.Count + 1):
        table = worksheet.ListObjects(index)
        table_last_row = table.Range.Row + table.Range.Rows.Count - 1
        if table_last_row != old_last_row or new_last_row <= old_last_row:
            continue
        first_column = table.Range.Column
        last_column = first_column + table.Range.Columns.Count - 1
        table.Resize(worksheet.Range(worksheet.Cells(table.Range.Row, first_column), worksheet.Cells(new_last_row, last_column)))
        return


def append_new_rows(worksheet: Any, master_headers: dict[str, int], rows: list[SourceRow]) -> None:
    if not rows:
        return
    old_last_row = last_used_row_in_column(worksheet, required_column(master_headers, DOCUMENT_ID_HEADER, worksheet.Name))
    first_target_row = max(2, old_last_row + 1)
    last_target_row = first_target_row + len(rows) - 1

    entity_column = required_column(master_headers, ENTITY_HEADER, worksheet.Name)
    worksheet.Range(worksheet.Cells(first_target_row, entity_column), worksheet.Cells(last_target_row, entity_column)).Value2 = tuple((row.entity,) for row in rows)
    for master_header, source_header in SOURCE_TO_MASTER_HEADERS.items():
        master_column = required_column(master_headers, master_header, worksheet.Name)
        source_key = normalize_header(source_header)
        values = tuple((row.values.get(source_key),) for row in rows)
        worksheet.Range(worksheet.Cells(first_target_row, master_column), worksheet.Cells(last_target_row, master_column)).Value2 = values

    for column in range(1, max(master_headers.values()) + 1):
        template_formula = worksheet.Cells(old_last_row, column).FormulaR1C1
        if isinstance(template_formula, str) and template_formula.startswith("="):
            worksheet.Range(worksheet.Cells(first_target_row, column), worksheet.Cells(last_target_row, column)).FormulaR1C1 = template_formula
    resize_containing_table(worksheet, old_last_row, last_target_row)


def run_sap_refresh() -> Path:
    if not IN_SCOPE_COST_CENTERS:
        raise ValueError("Set IN_SCOPE_COST_CENTERS before running the SAP refresh.")
    required_paths = (MASTER_FILE, BSNY_SAP_FILE, SANCAP_SAP_FILE)
    missing_paths = [str(path) for path in required_paths if not path.is_file()]
    if missing_paths:
        raise FileNotFoundError(f"Missing required workbook(s): {', '.join(missing_paths)}")

    excel = None
    master_book = bsny_book = sancap_book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        excel.Calculation = XL_CALCULATION_MANUAL

        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        bsny_book = excel.Workbooks.Open(str(BSNY_SAP_FILE.resolve()), ReadOnly=True)
        sancap_book = excel.Workbooks.Open(str(SANCAP_SAP_FILE.resolve()), ReadOnly=True)
        master_sheet = master_book.Worksheets(MASTER_SHEET_NAME)
        bsny_sheet = bsny_book.Worksheets(BSNY_SHEET_NAME)
        if sancap_book.Worksheets.Count != 1:
            raise ValueError("The SANCAP report must contain exactly one worksheet.")
        sancap_sheet = sancap_book.Worksheets(1)

        master_header_map, existing_master_rows = master_rows(master_sheet)
        validate_structure(master_header_map, read_headers(bsny_sheet))
        validate_structure(master_header_map, read_headers(sancap_sheet))
        source_rows = [
            *worksheet_rows(bsny_sheet, "BSNY", BSNY_SAP_FILE.name, REPORTING_YEAR),
            *worksheet_rows(sancap_sheet, "SANCAP", SANCAP_SAP_FILE.name, REPORTING_YEAR),
        ]
        existing_keys = {row.key for row in existing_master_rows if row.key is not None}
        classification = classify_source_rows(source_rows, existing_keys)
        failures = control_failures(source_rows, classification, existing_master_rows)
        audit_path = write_audit(classification, failures, source_rows, existing_master_rows)
        if failures:
            raise RuntimeError(f"SAP refresh controls failed. No master was saved. Audit: {audit_path}")

        master_book.Close(SaveChanges=False)
        master_book = None
        output_path = next_available_output_path(PUBLISHED_OUTPUT_FILE)
        shutil.copy2(MASTER_FILE, output_path)
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        master_sheet = master_book.Worksheets(MASTER_SHEET_NAME)
        append_new_rows(master_sheet, master_header_map, classification["new"])
        excel.Calculation = XL_CALCULATION_AUTOMATIC
        excel.CalculateFull()
        master_book.Save()
        master_book.Close(SaveChanges=True)
        master_book = None
        print(f"Appended {len(classification['new'])} new SAP row(s). Audit: {audit_path}")
        return output_path
    finally:
        for workbook in (master_book, bsny_book, sancap_book):
            if workbook is not None:
                workbook.Close(SaveChanges=False)
        if excel is not None:
            excel.Calculation = XL_CALCULATION_AUTOMATIC
            excel.Quit()
        pythoncom.CoUninitialize()


print("Run run_sap_refresh() after completing the configuration cell.")

Run run_sap_refresh() after completing the configuration cell.


## Load In-Scope Cost Centers

Reads the authoritative `SAP LA CC` values from the first table in the master workbook's `In Scope CCs` tab. The scan stops at the first blank row after that table, so any later section such as `Uber LOB Mapping` is ignored.

In [19]:
IN_SCOPE_COST_CENTERS_SHEET_NAME = "In Scope CCs"
IN_SCOPE_CC_NAME_HEADER = "CC Name"
IN_SCOPE_SAP_LA_CC_HEADER = "SAP LA CC"


def is_blank(value: Any) -> bool:
    return value is None or (isinstance(value, str) and not value.strip())


def load_in_scope_cost_centers(master_workbook: Any) -> set[str]:
    """Load SAP LA CCs from the first table in the master workbook's In Scope CCs tab."""
    worksheet = master_workbook.Worksheets(IN_SCOPE_COST_CENTERS_SHEET_NAME)
    used_range = worksheet.UsedRange
    first_row = used_range.Row
    last_row = first_row + used_range.Rows.Count - 1
    first_column = used_range.Column
    last_column = first_column + used_range.Columns.Count - 1
    values = worksheet.Range(
        worksheet.Cells(first_row, first_column), worksheet.Cells(last_row, last_column)
    ).Value2
    if not isinstance(values, tuple):
        values = ((values,),)
    elif values and not isinstance(values[0], tuple):
        values = (values,)

    header_row_index = None
    cc_name_column_index = None
    sap_la_cc_column_index = None
    for row_index, row_values in enumerate(values):
        normalized_headers = [normalize_header(value) for value in row_values]
        if (
            normalize_header(IN_SCOPE_CC_NAME_HEADER) in normalized_headers
            and normalize_header(IN_SCOPE_SAP_LA_CC_HEADER) in normalized_headers
        ):
            header_row_index = row_index
            cc_name_column_index = normalized_headers.index(normalize_header(IN_SCOPE_CC_NAME_HEADER))
            sap_la_cc_column_index = normalized_headers.index(normalize_header(IN_SCOPE_SAP_LA_CC_HEADER))
            break

    if header_row_index is None or cc_name_column_index is None or sap_la_cc_column_index is None:
        raise ValueError(
            f"Could not find the first in-scope table with {IN_SCOPE_CC_NAME_HEADER!r} "
            f"and {IN_SCOPE_SAP_LA_CC_HEADER!r} headers on {IN_SCOPE_COST_CENTERS_SHEET_NAME!r}."
        )

    cost_centers: set[str] = set()
    for row_values in values[header_row_index + 1:]:
        if all(is_blank(value) for value in row_values):
            break
        cost_center = normalize_cost_center(row_values[sap_la_cc_column_index])
        if cost_center is not None:
            cost_centers.add(cost_center)

    if not cost_centers:
        header_row = first_row + header_row_index
        raise ValueError(
            f"No SAP LA CC values were found below row {header_row} on "
            f"{IN_SCOPE_COST_CENTERS_SHEET_NAME!r}."
        )
    return cost_centers

In [20]:
_run_sap_refresh_without_post_save_validation = run_sap_refresh


def validate_published_output(output_path: Path) -> None:
    """Reopen the saved output and verify the master contains unique invoice keys."""
    excel = None
    workbook = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        workbook = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=True)
        _, rows = master_rows(workbook.Worksheets(MASTER_SHEET_NAME))
        key_counts = Counter(row.key for row in rows if row.key is not None)
        duplicates = [key for key, count in key_counts.items() if count > 1]
        if duplicates:
            raise RuntimeError(
                f"Saved output contains {len(duplicates)} duplicate Entity + Document ID key(s): {duplicates[:5]}"
            )
    finally:
        if workbook is not None:
            workbook.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


def run_sap_refresh() -> Path:
    run_output_path = _run_sap_refresh_without_post_save_validation()
    validate_published_output(run_output_path)
    if run_output_path != PUBLISHED_OUTPUT_FILE:
        shutil.copy2(run_output_path, PUBLISHED_OUTPUT_FILE)
    print(f"Post-save validation passed: {PUBLISHED_OUTPUT_FILE}")
    return PUBLISHED_OUTPUT_FILE

## Master Entity Mapping

The SAP master stores entities in `Company Code`, rather than an `Entity` column. This mapping converts `1428` to `SANCAP` and `6652` to `BSNY` for duplicate checks, then writes the corresponding company code when new records are appended.

In [21]:
ENTITY_HEADER = "Company Code"
ENTITY_TO_COMPANY_CODE = {
    "BSNY": "6652",
    "SANCAP": "1428",
}
COMPANY_CODE_TO_ENTITY = {
    company_code: entity for entity, company_code in ENTITY_TO_COMPANY_CODE.items()
}


def master_rows(worksheet: Any) -> tuple[dict[str, int], list[MasterRow]]:
    headers = read_headers(worksheet)
    entity_column = required_column(headers, ENTITY_HEADER, worksheet.Name)
    document_id_column = required_column(headers, DOCUMENT_ID_HEADER, worksheet.Name)
    posting_date_column = required_column(headers, POSTING_DATE_HEADER, worksheet.Name)
    cost_center_column = required_column(headers, COST_CENTER_HEADER, worksheet.Name)
    gross_amount_column = required_column(headers, GROSS_AMOUNT_HEADER, worksheet.Name)
    final_row = last_used_row_in_column(worksheet, document_id_column)
    final_column = max(headers.values())
    if final_row < 2:
        return headers, []

    raw_rows = worksheet.Range(worksheet.Cells(2, 1), worksheet.Cells(final_row, final_column)).Value2
    if not isinstance(raw_rows, tuple):
        raw_rows = (raw_rows,)
    if raw_rows and not isinstance(raw_rows[0], tuple):
        raw_rows = (raw_rows,)

    rows: list[MasterRow] = []
    for row_number, raw_row in enumerate(raw_rows, start=2):
        document_id = normalize_document_id(raw_row[document_id_column - 1])
        company_code = normalize_document_id(raw_row[entity_column - 1])
        entity = COMPANY_CODE_TO_ENTITY.get(company_code)
        if document_id is not None and company_code is not None and entity is None:
            raise ValueError(
                f"Unsupported Company Code {company_code!r} in {worksheet.Name!r}, row {row_number}."
            )
        key = (entity, document_id) if entity and document_id else None
        rows.append(
            MasterRow(
                key=key,
                posting_date=coerce_excel_date(raw_row[posting_date_column - 1]),
                cost_center=normalize_cost_center(raw_row[cost_center_column - 1]),
                gross_amount=decimal_amount(raw_row[gross_amount_column - 1]),
                row_number=row_number,
            )
        )
    return headers, rows


def append_new_rows(worksheet: Any, master_headers: dict[str, int], rows: list[SourceRow]) -> None:
    if not rows:
        return
    old_last_row = last_used_row_in_column(
        worksheet, required_column(master_headers, DOCUMENT_ID_HEADER, worksheet.Name)
    )
    first_target_row = max(2, old_last_row + 1)
    last_target_row = first_target_row + len(rows) - 1

    entity_column = required_column(master_headers, ENTITY_HEADER, worksheet.Name)
    company_codes = []
    for row in rows:
        company_code = ENTITY_TO_COMPANY_CODE.get(row.entity)
        if company_code is None:
            raise ValueError(f"No Company Code mapping is configured for entity {row.entity!r}.")
        company_codes.append((company_code,))
    worksheet.Range(
        worksheet.Cells(first_target_row, entity_column),
        worksheet.Cells(last_target_row, entity_column),
    ).Value2 = tuple(company_codes)

    for master_header, source_header in SOURCE_TO_MASTER_HEADERS.items():
        master_column = required_column(master_headers, master_header, worksheet.Name)
        source_key = normalize_header(source_header)
        values = tuple((row.values.get(source_key),) for row in rows)
        worksheet.Range(
            worksheet.Cells(first_target_row, master_column),
            worksheet.Cells(last_target_row, master_column),
        ).Value2 = values

    for column in range(1, max(master_headers.values()) + 1):
        template_formula = worksheet.Cells(old_last_row, column).FormulaR1C1
        if isinstance(template_formula, str) and template_formula.startswith("="):
            worksheet.Range(
                worksheet.Cells(first_target_row, column),
                worksheet.Cells(last_target_row, column),
            ).FormulaR1C1 = template_formula
    resize_containing_table(worksheet, old_last_row, last_target_row)

In [22]:
XL_PASTE_FORMATS = -4122


def worksheet_rows(worksheet: Any, entity: str, source_name: str, reporting_year: int) -> list[SourceRow]:
    """Bulk-read one SAP worksheet and retain rows posted in the reporting year."""
    headers = read_headers(worksheet)
    posting_date_column = required_column(headers, POSTING_DATE_HEADER, worksheet.Name)
    document_id_column = required_column(headers, DOCUMENT_ID_HEADER, worksheet.Name)
    final_row = last_used_row(worksheet)
    final_column = max(headers.values())
    if final_row < 2:
        return []

    raw_rows = worksheet.Range(worksheet.Cells(2, 1), worksheet.Cells(final_row, final_column)).Value2
    if not isinstance(raw_rows, tuple):
        raw_rows = (raw_rows,)
    if raw_rows and not isinstance(raw_rows[0], tuple):
        raw_rows = (raw_rows,)

    retained: list[SourceRow] = []
    posting_date_key = normalize_header(POSTING_DATE_HEADER)
    for row_number, raw_row in enumerate(raw_rows, start=2):
        posting_date = coerce_excel_date(raw_row[posting_date_column - 1])
        if posting_date is None or posting_date.year != reporting_year:
            continue

        values = {header: raw_row[column - 1] for header, column in headers.items()}
        # Write a true date object so Excel retains an unambiguous Posting Date value.
        values[posting_date_key] = posting_date
        retained.append(
            SourceRow(
                entity=normalize_text(entity),
                document_id=normalize_document_id(raw_row[document_id_column - 1]),
                posting_date=posting_date,
                values=values,
                source_name=source_name,
                source_row_number=row_number,
            )
        )
    return retained


def append_new_rows(worksheet: Any, master_headers: dict[str, int], rows: list[SourceRow]) -> None:
    """Append all source fields shared with the master and preserve master formats/formulas."""
    if not rows:
        return

    document_id_column = required_column(master_headers, DOCUMENT_ID_HEADER, worksheet.Name)
    old_last_row = last_used_row_in_column(worksheet, document_id_column)
    first_target_row = max(2, old_last_row + 1)
    last_target_row = first_target_row + len(rows) - 1
    final_master_column = max(master_headers.values())

    # Match the display formats of the existing master row before writing values.
    template_range = worksheet.Range(
        worksheet.Cells(old_last_row, 1), worksheet.Cells(old_last_row, final_master_column)
    )
    target_range = worksheet.Range(
        worksheet.Cells(first_target_row, 1), worksheet.Cells(last_target_row, final_master_column)
    )
    template_range.Copy()
    target_range.PasteSpecial(Paste=XL_PASTE_FORMATS)
    worksheet.Application.CutCopyMode = False

    entity_column = required_column(master_headers, ENTITY_HEADER, worksheet.Name)
    company_codes = []
    for row in rows:
        company_code = ENTITY_TO_COMPANY_CODE.get(row.entity)
        if company_code is None:
            raise ValueError(f"No Company Code mapping is configured for entity {row.entity!r}.")
        company_codes.append((company_code,))
    worksheet.Range(
        worksheet.Cells(first_target_row, entity_column),
        worksheet.Cells(last_target_row, entity_column),
    ).Value2 = tuple(company_codes)

    # Every other matching master/source header is transferred in bulk. Helper-only
    # master columns are intentionally left for the formula-fill step below.
    for master_header, master_column in master_headers.items():
        if master_header == normalize_header(ENTITY_HEADER):
            continue
        if not any(master_header in row.values for row in rows):
            continue
        values = tuple((row.values.get(master_header),) for row in rows)
        worksheet.Range(
            worksheet.Cells(first_target_row, master_column),
            worksheet.Cells(last_target_row, master_column),
        ).Value2 = values

    for column in range(1, final_master_column + 1):
        template_formula = worksheet.Cells(old_last_row, column).FormulaR1C1
        if isinstance(template_formula, str) and template_formula.startswith("="):
            worksheet.Range(
                worksheet.Cells(first_target_row, column),
                worksheet.Cells(last_target_row, column),
            ).FormulaR1C1 = template_formula
    resize_containing_table(worksheet, old_last_row, last_target_row)

In [23]:
def worksheet_rows(worksheet: Any, entity: str, source_name: str, reporting_year: int) -> list[SourceRow]:
    """Bulk-read one SAP worksheet and retain rows posted in the reporting year."""
    headers = read_headers(worksheet)
    posting_date_column = required_column(headers, POSTING_DATE_HEADER, worksheet.Name)
    document_id_column = required_column(headers, DOCUMENT_ID_HEADER, worksheet.Name)
    final_row = last_used_row(worksheet)
    final_column = max(headers.values())
    if final_row < 2:
        return []

    raw_rows = worksheet.Range(worksheet.Cells(2, 1), worksheet.Cells(final_row, final_column)).Value2
    if not isinstance(raw_rows, tuple):
        raw_rows = (raw_rows,)
    if raw_rows and not isinstance(raw_rows[0], tuple):
        raw_rows = (raw_rows,)

    retained: list[SourceRow] = []
    posting_date_key = normalize_header(POSTING_DATE_HEADER)
    for row_number, raw_row in enumerate(raw_rows, start=2):
        posting_date = coerce_excel_date(raw_row[posting_date_column - 1])
        if posting_date is None or posting_date.year != reporting_year:
            continue

        values = {header: raw_row[column - 1] for header, column in headers.items()}
        # pywin32 requires a datetime, rather than a date-only value, for Excel ranges.
        values[posting_date_key] = dt.datetime.combine(posting_date, dt.time.min)
        retained.append(
            SourceRow(
                entity=normalize_text(entity),
                document_id=normalize_document_id(raw_row[document_id_column - 1]),
                posting_date=posting_date,
                values=values,
                source_name=source_name,
                source_row_number=row_number,
            )
        )
    return retained

In [24]:
VIM_PROCESS_STATUS_HEADER = "VIM Process Status Text"
POSTED_VIM_PROCESS_STATUS = "POSTED"
SOURCE_FILTER_COUNTS: dict[str, dict[str, int]] = {}


def worksheet_rows(worksheet: Any, entity: str, source_name: str, reporting_year: int) -> list[SourceRow]:
    """Bulk-read only reporting-year SAP rows whose VIM status is Posted."""
    headers = read_headers(worksheet)
    posting_date_column = required_column(headers, POSTING_DATE_HEADER, worksheet.Name)
    document_id_column = required_column(headers, DOCUMENT_ID_HEADER, worksheet.Name)
    vim_status_column = required_column(headers, VIM_PROCESS_STATUS_HEADER, worksheet.Name)
    final_row = last_used_row(worksheet)
    final_column = max(headers.values())
    if final_row < 2:
        SOURCE_FILTER_COUNTS[source_name] = {
            "in_year": 0,
            "excluded_non_posted": 0,
            "retained_posted": 0,
        }
        return []

    raw_rows = worksheet.Range(worksheet.Cells(2, 1), worksheet.Cells(final_row, final_column)).Value2
    if not isinstance(raw_rows, tuple):
        raw_rows = (raw_rows,)
    if raw_rows and not isinstance(raw_rows[0], tuple):
        raw_rows = (raw_rows,)

    retained: list[SourceRow] = []
    in_year_count = 0
    excluded_non_posted_count = 0
    posting_date_key = normalize_header(POSTING_DATE_HEADER)
    for row_number, raw_row in enumerate(raw_rows, start=2):
        posting_date = coerce_excel_date(raw_row[posting_date_column - 1])
        if posting_date is None or posting_date.year != reporting_year:
            continue
        in_year_count += 1

        vim_status = normalize_text(raw_row[vim_status_column - 1])
        if vim_status != POSTED_VIM_PROCESS_STATUS:
            excluded_non_posted_count += 1
            continue

        values = {header: raw_row[column - 1] for header, column in headers.items()}
        values[posting_date_key] = dt.datetime.combine(posting_date, dt.time.min)
        retained.append(
            SourceRow(
                entity=normalize_text(entity),
                document_id=normalize_document_id(raw_row[document_id_column - 1]),
                posting_date=posting_date,
                values=values,
                source_name=source_name,
                source_row_number=row_number,
            )
        )

    SOURCE_FILTER_COUNTS[source_name] = {
        "in_year": in_year_count,
        "excluded_non_posted": excluded_non_posted_count,
        "retained_posted": len(retained),
    }
    return retained

In [25]:
_posted_worksheet_rows = worksheet_rows


def worksheet_rows(worksheet: Any, entity: str, source_name: str, reporting_year: int) -> list[SourceRow]:
    rows = _posted_worksheet_rows(worksheet, entity, source_name, reporting_year)
    counts = SOURCE_FILTER_COUNTS[source_name]
    print(
        f"[SAP Refresh] {entity} source filter: in-year={counts['in_year']}, "
        f"excluded_non_posted={counts['excluded_non_posted']}, "
        f"retained_posted={counts['retained_posted']}"
    )
    return rows

In [26]:
def validate_structure(master_headers: dict[str, int], source_headers: dict[str, int]) -> None:
    """Verify required columns are present in the resolved master and one source report."""
    master_sheet_name = "resolved master SAP worksheet"
    required_column(master_headers, ENTITY_HEADER, master_sheet_name)
    for master_header, source_header in SOURCE_TO_MASTER_HEADERS.items():
        required_column(master_headers, master_header, master_sheet_name)
        required_column(source_headers, source_header, "source report")

In [27]:
def resolve_master_sheet(workbook: Any) -> Any:
    """Return the supported SAP worksheet found in the master workbook."""
    available_names = {
        normalize_text(workbook.Worksheets(index).Name): workbook.Worksheets(index)
        for index in range(1, workbook.Worksheets.Count + 1)
    }
    for sheet_name in MASTER_SHEET_NAMES:
        worksheet = available_names.get(normalize_text(sheet_name))
        if worksheet is not None:
            return worksheet
    found_names = [worksheet.Name for worksheet in available_names.values()]
    raise ValueError(
        "Master workbook must contain either "
        f"{MASTER_SHEET_NAMES[0]!r} or {MASTER_SHEET_NAMES[1]!r}. Found: {found_names}"
    )


def set_excel_calculation(excel: Any, calculation_mode: int, action: str) -> None:
    """Attempt an optional calculation-mode change without masking the refresh."""
    try:
        excel.Calculation = calculation_mode
        print(f"[SAP Refresh] Excel calculation mode set to {action}")
    except pythoncom.com_error as error:
        print(f"[SAP Refresh] Warning: could not {action}; continuing. {error}")


def close_workbook(workbook: Any, name: str) -> None:
    if workbook is None:
        return
    try:
        workbook.Close(SaveChanges=False)
        print(f"[SAP Refresh] Closed {name} workbook")
    except pythoncom.com_error as error:
        print(f"[SAP Refresh] Warning: could not close {name} workbook. {error}")


def validate_published_output(output_path: Path) -> None:
    """Reopen the saved output and verify the master contains unique invoice keys."""
    excel = None
    workbook = None
    pythoncom.CoInitialize()
    try:
        print(f"[SAP Refresh] Post-save validation: opening {output_path}")
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        workbook = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=True)
        _, rows = master_rows(resolve_master_sheet(workbook))
        key_counts = Counter(row.key for row in rows if row.key is not None)
        duplicates = [key for key, count in key_counts.items() if count > 1]
        if duplicates:
            raise RuntimeError(
                f"Saved output contains {len(duplicates)} duplicate Entity + Document ID key(s): {duplicates[:5]}"
            )
        print(f"[SAP Refresh] Post-save validation passed: {len(rows)} rows checked")
    finally:
        close_workbook(workbook, "post-save validation")
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


def run_sap_refresh() -> Path:
    global IN_SCOPE_COST_CENTERS

    required_paths = (MASTER_FILE, BSNY_SAP_FILE, SANCAP_SAP_FILE)
    missing_paths = [str(path) for path in required_paths if not path.is_file()]
    if missing_paths:
        raise FileNotFoundError(f"Missing required workbook(s): {', '.join(missing_paths)}")

    excel = None
    master_book = bsny_book = sancap_book = None
    stage = "initializing"
    pythoncom.CoInitialize()
    try:
        print(f"[SAP Refresh] Starting refresh for reporting year {REPORTING_YEAR}")
        stage = "starting isolated Excel COM instance"
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        set_excel_calculation(excel, XL_CALCULATION_MANUAL, "manual")

        stage = "opening master and source workbooks"
        print("[SAP Refresh] Opening master and source workbooks")
        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        bsny_book = excel.Workbooks.Open(str(BSNY_SAP_FILE.resolve()), ReadOnly=True)
        sancap_book = excel.Workbooks.Open(str(SANCAP_SAP_FILE.resolve()), ReadOnly=True)
        master_sheet = resolve_master_sheet(master_book)
        bsny_sheet = bsny_book.Worksheets(BSNY_SHEET_NAME)
        if sancap_book.Worksheets.Count != 1:
            raise ValueError("The SANCAP report must contain exactly one worksheet.")
        sancap_sheet = sancap_book.Worksheets(1)
        print(
            f"[SAP Refresh] Resolved sheets: master={master_sheet.Name}, "
            f"BSNY={bsny_sheet.Name}, SANCAP={sancap_sheet.Name}"
        )

        stage = "loading in-scope SAP LA CC values"
        IN_SCOPE_COST_CENTERS = load_in_scope_cost_centers(master_book)
        print(
            f"[SAP Refresh] Loaded {len(IN_SCOPE_COST_CENTERS)} SAP LA CC value(s) "
            f"from the first table on {IN_SCOPE_COST_CENTERS_SHEET_NAME!r}"
        )

        stage = "validating headers and reading master data"
        master_header_map, existing_master_rows = master_rows(master_sheet)
        validate_structure(master_header_map, read_headers(bsny_sheet))
        validate_structure(master_header_map, read_headers(sancap_sheet))
        print(f"[SAP Refresh] Header validation passed; master rows={len(existing_master_rows)}")

        stage = "extracting reporting-year source rows"
        bsny_rows = worksheet_rows(bsny_sheet, "BSNY", BSNY_SAP_FILE.name, REPORTING_YEAR)
        sancap_rows = worksheet_rows(sancap_sheet, "SANCAP", SANCAP_SAP_FILE.name, REPORTING_YEAR)
        source_rows = [*bsny_rows, *sancap_rows]
        print(
            f"[SAP Refresh] Extracted in-year rows: BSNY={len(bsny_rows)}, "
            f"SANCAP={len(sancap_rows)}"
        )

        stage = "classifying source rows and running controls"
        existing_keys = {row.key for row in existing_master_rows if row.key is not None}
        classification = classify_source_rows(source_rows, existing_keys)
        print(
            "[SAP Refresh] Classification: "
            f"new={len(classification['new'])}, "
            f"existing={len(classification['existing'])}, "
            f"missing_id={len(classification['missing_id'])}, "
            f"duplicates={len(classification['duplicate_in_source'])}"
        )

        failures = control_failures(source_rows, classification, existing_master_rows)
        audit_path = write_audit(classification, failures, source_rows, existing_master_rows)
        print(f"[SAP Refresh] Audit written: {audit_path}")
        if failures:
            raise RuntimeError(f"SAP refresh controls failed. No master was saved. Audit: {audit_path}")

        stage = "creating and appending the run output"
        close_workbook(master_book, "original master")
        master_book = None
        output_path = next_available_output_path(PUBLISHED_OUTPUT_FILE)
        print(f"[SAP Refresh] Creating run output: {output_path}")
        shutil.copy2(MASTER_FILE, output_path)
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        master_sheet = resolve_master_sheet(master_book)
        append_new_rows(master_sheet, master_header_map, classification["new"])
        print(f"[SAP Refresh] Appended {len(classification['new'])} new row(s)")

        stage = "recalculating and saving the run output"
        set_excel_calculation(excel, XL_CALCULATION_AUTOMATIC, "automatic")
        excel.CalculateFull()
        master_book.Save()
        master_book.Close(SaveChanges=True)
        master_book = None
        print(f"[SAP Refresh] Saved run output: {output_path}")

        # Release every handle from the write COM instance before validation opens the file.
        close_workbook(bsny_book, "BSNY")
        bsny_book = None
        close_workbook(sancap_book, "SANCAP")
        sancap_book = None
        set_excel_calculation(excel, XL_CALCULATION_AUTOMATIC, "automatic during validation cleanup")
        excel.Quit()
        excel = None
        print("[SAP Refresh] Write Excel COM instance closed before post-save validation")

        stage = "validating and publishing the saved output"
        validate_published_output(output_path)
        if output_path != PUBLISHED_OUTPUT_FILE:
            shutil.copy2(output_path, PUBLISHED_OUTPUT_FILE)
        print(f"[SAP Refresh] Completed successfully: {PUBLISHED_OUTPUT_FILE}")
        return PUBLISHED_OUTPUT_FILE
    except Exception as error:
        print(f"[SAP Refresh] Failed while {stage}: {type(error).__name__}: {error}")
        raise
    finally:
        close_workbook(master_book, "master")
        close_workbook(bsny_book, "BSNY")
        close_workbook(sancap_book, "SANCAP")
        if excel is not None:
            set_excel_calculation(excel, XL_CALCULATION_AUTOMATIC, "automatic during cleanup")
            try:
                excel.Quit()
                print("[SAP Refresh] Excel COM instance closed")
            except pythoncom.com_error as error:
                print(f"[SAP Refresh] Warning: could not close Excel COM instance. {error}")
        pythoncom.CoUninitialize()


print("Active SAP refresh workflow loaded with resilient Excel calculation handling.")

Active SAP refresh workflow loaded with resilient Excel calculation handling.


## Post-Save Validation

Uses a read-only workbook reader for the final duplicate-key check after Excel COM saves the refreshed file. This avoids Excel retaining a short-lived file lock after `Quit()`.

In [28]:
from openpyxl import load_workbook


def validate_published_output(output_path: Path) -> None:
    """Read the saved output without Excel COM and verify master invoice keys are unique."""
    print(f"[SAP Refresh] Post-save validation: reading {output_path}")
    workbook = load_workbook(output_path, read_only=True, data_only=False)
    try:
        worksheet = next(
            (
                workbook[sheet_name]
                for sheet_name in workbook.sheetnames
                if normalize_text(sheet_name) in {normalize_text(name) for name in MASTER_SHEET_NAMES}
            ),
            None,
        )
        if worksheet is None:
            raise ValueError(
                f"Saved output must contain one of {MASTER_SHEET_NAMES}; found {workbook.sheetnames}."
            )

        raw_headers = next(worksheet.iter_rows(min_row=1, max_row=1, values_only=True))
        headers = {
            normalize_header(header): column_number
            for column_number, header in enumerate(raw_headers, start=1)
            if normalize_header(header)
        }
        entity_column = required_column(headers, ENTITY_HEADER, worksheet.title)
        document_id_column = required_column(headers, DOCUMENT_ID_HEADER, worksheet.title)

        key_counts: Counter[tuple[str, str]] = Counter()
        for row_values in worksheet.iter_rows(min_row=2, values_only=True):
            document_id = normalize_document_id(row_values[document_id_column - 1])
            company_code = normalize_document_id(row_values[entity_column - 1])
            entity = COMPANY_CODE_TO_ENTITY.get(company_code)
            if document_id is not None and company_code is not None and entity is None:
                raise ValueError(
                    f"Saved output contains unsupported Company Code {company_code!r}."
                )
            if entity and document_id:
                key_counts[(entity, document_id)] += 1

        duplicates = [key for key, count in key_counts.items() if count > 1]
        if duplicates:
            raise RuntimeError(
                f"Saved output contains {len(duplicates)} duplicate Entity + Document ID key(s): {duplicates[:5]}"
            )
        print(f"[SAP Refresh] Post-save validation passed: {sum(key_counts.values())} invoice keys checked")
    finally:
        workbook.close()

In [29]:
from pathlib import Path

In [30]:
import gc
import hashlib


def _file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as workbook_file:
        for chunk in iter(lambda: workbook_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def publish_validated_output(run_output_path: Path) -> Path:
    """Publish the validated run output and prove the canonical file is identical."""
    if run_output_path != PUBLISHED_OUTPUT_FILE:
        shutil.copy2(run_output_path, PUBLISHED_OUTPUT_FILE)
        if _file_sha256(run_output_path) != _file_sha256(PUBLISHED_OUTPUT_FILE):
            raise RuntimeError(
                f"Published workbook differs from validated run output: {PUBLISHED_OUTPUT_FILE}"
            )
    validate_published_output(PUBLISHED_OUTPUT_FILE)
    return PUBLISHED_OUTPUT_FILE


def run_sap_refresh() -> Path:
    """Refresh the SAP master and release Excel COM handles before final validation."""
    global IN_SCOPE_COST_CENTERS

    required_paths = (MASTER_FILE, BSNY_SAP_FILE, SANCAP_SAP_FILE)
    missing_paths = [str(path) for path in required_paths if not path.is_file()]
    if missing_paths:
        raise FileNotFoundError(f"Missing required workbook(s): {', '.join(missing_paths)}")

    excel = None
    master_book = bsny_book = sancap_book = None
    master_sheet = bsny_sheet = sancap_sheet = None
    stage = "initializing"
    pythoncom.CoInitialize()
    try:
        print(f"[SAP Refresh] Starting refresh for reporting year {REPORTING_YEAR}")
        stage = "starting isolated Excel COM instance"
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        set_excel_calculation(excel, XL_CALCULATION_MANUAL, "manual")

        stage = "opening master and source workbooks"
        print("[SAP Refresh] Opening master and source workbooks")
        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        bsny_book = excel.Workbooks.Open(str(BSNY_SAP_FILE.resolve()), ReadOnly=True)
        sancap_book = excel.Workbooks.Open(str(SANCAP_SAP_FILE.resolve()), ReadOnly=True)
        master_sheet = resolve_master_sheet(master_book)
        bsny_sheet = bsny_book.Worksheets(BSNY_SHEET_NAME)
        if sancap_book.Worksheets.Count != 1:
            raise ValueError("The SANCAP report must contain exactly one worksheet.")
        sancap_sheet = sancap_book.Worksheets(1)
        print(
            f"[SAP Refresh] Resolved sheets: master={master_sheet.Name}, "
            f"BSNY={bsny_sheet.Name}, SANCAP={sancap_sheet.Name}"
        )

        stage = "loading in-scope SAP LA CC values"
        IN_SCOPE_COST_CENTERS = load_in_scope_cost_centers(master_book)
        print(
            f"[SAP Refresh] Loaded {len(IN_SCOPE_COST_CENTERS)} SAP LA CC value(s) "
            f"from the first table on {IN_SCOPE_COST_CENTERS_SHEET_NAME!r}"
        )

        stage = "validating headers and reading master data"
        master_header_map, existing_master_rows = master_rows(master_sheet)
        validate_structure(master_header_map, read_headers(bsny_sheet))
        validate_structure(master_header_map, read_headers(sancap_sheet))
        print(f"[SAP Refresh] Header validation passed; master rows={len(existing_master_rows)}")

        stage = "extracting reporting-year source rows"
        bsny_rows = worksheet_rows(bsny_sheet, "BSNY", BSNY_SAP_FILE.name, REPORTING_YEAR)
        sancap_rows = worksheet_rows(sancap_sheet, "SANCAP", SANCAP_SAP_FILE.name, REPORTING_YEAR)
        source_rows = [*bsny_rows, *sancap_rows]
        print(
            f"[SAP Refresh] Extracted in-year rows: BSNY={len(bsny_rows)}, "
            f"SANCAP={len(sancap_rows)}"
        )

        stage = "classifying source rows and running controls"
        existing_keys = {row.key for row in existing_master_rows if row.key is not None}
        classification = classify_source_rows(source_rows, existing_keys)
        print(
            "[SAP Refresh] Classification: "
            f"new={len(classification['new'])}, existing={len(classification['existing'])}, "
            f"missing_id={len(classification['missing_id'])}, "
            f"duplicates={len(classification['duplicate_in_source'])}"
        )
        failures = control_failures(source_rows, classification, existing_master_rows)
        audit_path = write_audit(classification, failures, source_rows, existing_master_rows)
        print(f"[SAP Refresh] Audit written: {audit_path}")
        if failures:
            raise RuntimeError(f"SAP refresh controls failed. No master was saved. Audit: {audit_path}")

        stage = "creating and appending the run output"
        close_workbook(master_book, "original master")
        master_book = None
        master_sheet = None
        output_path = next_available_output_path(PUBLISHED_OUTPUT_FILE)
        print(f"[SAP Refresh] Creating run output: {output_path}")
        shutil.copy2(MASTER_FILE, output_path)
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        master_sheet = resolve_master_sheet(master_book)
        append_new_rows(master_sheet, master_header_map, classification["new"])
        print(f"[SAP Refresh] Appended {len(classification['new'])} new row(s)")

        stage = "recalculating and saving the run output"
        set_excel_calculation(excel, XL_CALCULATION_AUTOMATIC, "automatic")
        excel.CalculateFull()
        master_book.Save()
        master_book.Close(SaveChanges=True)
        master_book = None
        master_sheet = None
        print(f"[SAP Refresh] Saved run output: {output_path}")

        stage = "releasing Excel COM handles before post-save validation"
        bsny_sheet = None
        sancap_sheet = None
        close_workbook(bsny_book, "BSNY")
        bsny_book = None
        close_workbook(sancap_book, "SANCAP")
        sancap_book = None
        set_excel_calculation(excel, XL_CALCULATION_AUTOMATIC, "automatic during validation cleanup")
        excel.Quit()
        excel = None
        gc.collect()
        print("[SAP Refresh] Write Excel COM instance and worksheet handles released")

        stage = "validating and publishing the saved output"
        validate_published_output(output_path)
        published_output_path = publish_validated_output(output_path)
        print(f"[SAP Refresh] Completed successfully: {published_output_path}")
        return published_output_path
    except Exception as error:
        print(f"[SAP Refresh] Failed while {stage}: {type(error).__name__}: {error}")
        raise
    finally:
        master_sheet = bsny_sheet = sancap_sheet = None
        close_workbook(master_book, "master")
        close_workbook(bsny_book, "BSNY")
        close_workbook(sancap_book, "SANCAP")
        if excel is not None:
            set_excel_calculation(excel, XL_CALCULATION_AUTOMATIC, "automatic during cleanup")
            try:
                excel.Quit()
                print("[SAP Refresh] Excel COM instance closed")
            except pythoncom.com_error as error:
                print(f"[SAP Refresh] Warning: could not close Excel COM instance. {error}")
        gc.collect()
        pythoncom.CoUninitialize()


print("Active SAP refresh workflow loaded with COM handle release before validation.")

Active SAP refresh workflow loaded with COM handle release before validation.


In [31]:
_full_calculation_refresh = run_sap_refresh


class _ExcelCalculationProxy:
    """Use normal dirty-formula calculation in place of an expensive full rebuild."""

    def __init__(self, application: Any) -> None:
        object.__setattr__(self, "_application", application)

    def __getattr__(self, name: str) -> Any:
        return getattr(self._application, name)

    def __setattr__(self, name: str, value: Any) -> None:
        setattr(self._application, name, value)

    def CalculateFull(self) -> None:
        print("[SAP Refresh] Calculating changed formulas only")
        self._application.Calculate()


def run_sap_refresh() -> Path:
    """Run the validated SAP refresh using normal formula recalculation for speed."""
    original_dispatch = win32.DispatchEx

    def dispatch_excel(*args: Any, **kwargs: Any) -> _ExcelCalculationProxy:
        return _ExcelCalculationProxy(original_dispatch(*args, **kwargs))

    win32.DispatchEx = dispatch_excel
    try:
        return _full_calculation_refresh()
    finally:
        win32.DispatchEx = original_dispatch


print("Fast SAP refresh wrapper loaded: changed formulas will be calculated without a full workbook rebuild.")

Fast SAP refresh wrapper loaded: changed formulas will be calculated without a full workbook rebuild.


In [36]:
refreshed_file = run_sap_refresh()
print(refreshed_file)

[SAP Refresh] Starting refresh for reporting year 2026
[SAP Refresh] Warning: could not manual; continuing. (-2147352567, 'Exception occurred.', (0, 'Microsoft Excel', 'Unable to set the Calculation property of the Application class', 'xlmain11.chm', 0, -2146827284), None)
[SAP Refresh] Opening master and source workbooks
[SAP Refresh] Resolved sheets: master=SAP Report, BSNY=SAP, SANCAP=Data
[SAP Refresh] Loaded 16 SAP LA CC value(s) from the first table on 'In Scope CCs'
[SAP Refresh] Header validation passed; master rows=2605
[SAP Refresh] BSNY source filter: in-year=138, excluded_non_posted=4, retained_posted=134
[SAP Refresh] SANCAP source filter: in-year=3452, excluded_non_posted=400, retained_posted=3052
[SAP Refresh] Extracted in-year rows: BSNY=134, SANCAP=3052
[SAP Refresh] Classification: new=585, existing=2601, missing_id=0, duplicates=0
[SAP Refresh] Audit written: outputs\sap_refresh_audit_20260818_085918.csv
[SAP Refresh] Closed original master workbook
[SAP Refresh] Cre